# Gộp dữ liệu + bảng retrieval thành dataset hoàn chỉnh (Kaggle)

**Input** (*Add Input → Your Work → Notebooks*):
- output notebook dựng dữ liệu (`kaggle_build_public_data.ipynb`) của từng tập: `ms-pred/data/spec_datasets/<tập>/` có
  `labels.tsv`, `spec_files.hdf5`, `splits/`, `magma_outputs/`, `subformulae/`;
- output notebook retrieval (`kaggle_pubchem_candidates.ipynb`, ví dụ `retrieval_merge`):
  `ms-pred/data/spec_datasets/<tập>/retrieval/cands_df_split_1_50.tsv`.

Notebook chép dữ liệu của mỗi tập ra `/kaggle/working/ms-pred/data/spec_datasets/<tập>/`, thêm bảng retrieval vào
`retrieval/`, rồi kiểm tra. Không cần GPU, không cần Internet, không cài gì.

Output dùng thẳng cho notebook train (`RESUME_FROM = "/kaggle/input/<tên notebook này>/ms-pred"`) hoặc tải về server
(chép `ms-pred/data/spec_datasets/<tập>/` vào `~/ms-pred/data/spec_datasets/`).

`/kaggle/working` tối đa 20 GB: nếu báo đầy đĩa, chạy riêng từng tập (`DATASETS = ["msnlib"]`, rồi một bản khác cho massbank).

In [ ]:
DATASETS = ["massbank", "msnlib"]     # tập cần gộp
INPUTS   = ["/kaggle/input"]          # nơi tìm output các notebook (tìm đệ quy)

In [ ]:
import json, shutil
from pathlib import Path
import pandas as pd

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REQUIRED = ["labels.tsv", "spec_files.hdf5", "magma_outputs/magma_tree.hdf5", "subformulae/no_subform.hdf5"]

def find_data_dir(ds):
    """Thư mục spec_datasets/<ds> có đủ dữ liệu (output notebook dựng dữ liệu)."""
    cands = [p.parent for root in INPUTS for p in Path(root).glob(f"**/spec_datasets/{ds}/labels.tsv")
             if all((p.parent / f).exists() for f in REQUIRED)]
    if not cands:
        raise SystemExit(f"Không thấy dữ liệu {ds} (labels.tsv + spec_files.hdf5 + magma_outputs + subformulae) trong {INPUTS}")
    if len(cands) > 1:
        print(f"  {ds}: có {len(cands)} bản dữ liệu, dùng {cands[0]}  (các bản khác: {[str(c) for c in cands[1:]]})")
    return cands[0]

def find_retrieval(ds, data_dir):
    """Bảng ứng viên của ds nằm ngoài thư mục dữ liệu (output notebook retrieval)."""
    tables = {}
    for root in INPUTS:
        for p in Path(root).glob(f"**/spec_datasets/{ds}/retrieval/cands_df_*.tsv"):
            if data_dir not in p.parents:
                tables.setdefault(p.name, p)
    return tables

summary = {}
for ds in DATASETS:
    print(f"\n===== {ds} =====")
    src = find_data_dir(ds)
    tables = find_retrieval(ds, src)
    if not tables:
        raise SystemExit(f"Không thấy bảng retrieval cho {ds}: Add Input output notebook retrieval")
    dst = WORK / "ms-pred/data/spec_datasets" / ds
    if dst.exists():
        shutil.rmtree(dst)
    print(f"Chép dữ liệu {src} -> {dst}", flush=True)
    shutil.copytree(src, dst, symlinks=False)
    (dst / "retrieval").mkdir(exist_ok=True)
    for name, p in sorted(tables.items()):
        if (dst / "retrieval" / name).exists():
            print(f"  thay {name} có sẵn bằng bản từ {p.parent}")
        shutil.copy2(p, dst / "retrieval" / name)
        print(f"  + retrieval/{name}  ({p})")

    # Kiểm tra: phổ trong bảng ứng viên phải có trong labels.tsv và thuộc tập test
    labels = pd.read_csv(dst / "labels.tsv", sep="\t")
    split = pd.read_csv(dst / "splits/split_1.tsv", sep="\t")
    name_col, fold_col = split.columns[:2]
    test = set(split.loc[split[fold_col] == "test", name_col].astype(str))
    info = {"entries": len(labels), "test_entries": len(test)}
    for name in sorted(tables):
        df = pd.read_csv(dst / "retrieval" / name, sep="\t")
        specs = set(df["spec"].astype(str))
        per = df.groupby("spec").size()
        not_label = specs - set(labels["spec"].astype(str))
        target = test if "_val_" not in name else set(split.loc[split[fold_col] == "val", name_col].astype(str))
        info[name] = {"rows": len(df), "spectra": len(specs), "fold_spectra": len(target),
                      "coverage": f"{100 * len(specs & target) / max(len(target), 1):.1f}%",
                      "candidates_median": float(per.median()), "candidates_max": int(per.max()),
                      "not_in_labels": len(not_label), "not_in_fold": len(specs - target)}
        if not_label or specs - target or per.max() > 50:
            print(f"  CẢNH BÁO {name}: {len(not_label)} phổ không có trong labels, {len(specs - target)} phổ ngoài fold, "
                  f"tối đa {per.max()} ứng viên")
    missing = [f for f in REQUIRED + ["splits/split_1.tsv"] if not (dst / f).exists()]
    info["missing_files"] = missing
    summary[ds] = info
    print(json.dumps(info, indent=2, ensure_ascii=False))
    print("Dung lượng:", f"{sum(f.stat().st_size for f in dst.rglob('*') if f.is_file()) / 2**30:.2f} GB")

(WORK / "merge_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
ok = all(not v["missing_files"] for v in summary.values())
print("\n" + ("XONG: Save Version -> Save & Run All, rồi tab Output -> New Dataset (Public)" if ok else "THIẾU FILE, xem ở trên"))